# TeleRAG-Agent: QLoRA Fine-Tuning on Kaggle (T4)
This notebook uses **Unsloth** for 2-5x faster training and 80% less memory usage.
**Fixed for AttributeError: `int` object has no attribute `mean`**

In [ ]:
!pip install unsloth transformers datasets accelerate bitsandbytes xformers -q

In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFimport torch
import gc
from torch.utils.data import DataLoader
from unsloth import FastLanguageModel
from datasets import load_dataset
from transformers import get_linear_schedule_with_warmup
from tqdm import tqdm
from kaggle_secrets import UserSecretsClient

max_seq_length = 2048
dtype = None
load_in_4bit = True

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
except:
    hf_token = ""TTrainer
from transformers import TrainingArguments
from kaggle_secrets import UserSecretsClient

# ----- Configuration -----
max_seq_length = 2048
dtype = None
load_in_4bit = True

# ----- Load HF Token -----
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
except:
    hf_token = "" # Fallback if not running in Kaggle environment with secrets

In [ ]:
model_name = "AliMaatouk/LLama-3-8B-Tele-it"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    token=hf_token,
)

# Critical: set pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False  # Important for gradient checkpointing

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
dataset = load_dataset(
    "json",
    data_files="/kaggle/input/datasets/chaitanyakadupukutla/new-kaggle-tele/teleqna_train.jsonl",
    split="train"
)

def formatting_prompts_func(examples):
    texts = []
    for inp, out in zip(examples['input'], examples['output']):
        if not inp or not out:
            continue
        # Simple clear format
        text = f"### Question:\n{inp}\n\n### Answer:\n{out}"
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True, remove_columns=dataset.column_names)
print("Example:", dataset[0]['text'][:300])

### 100-Sample Test Run (Run this first!)
Verifies that everything works without OOM before doing the full multi-hour training loop.

In [ ]:
# We will tokenize the full text, but then create labels where only the answer tokens are kept.
# First, get the token IDs for the answer separator.
answer_sep = "### Answer:\n"
answer_sep_ids = tokenizer.encode(answer_sep, add_special_tokens=False)
print(f"Answer separator token IDs: {answer_sep_ids}")

def tokenize_and_mask(example):
    # Tokenize full text
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        max_length=max_seq_length,
        padding=False,
        return_tensors=None,
    )
    input_ids = tokenized["input_ids"]
    attention_mask = tokenized["attention_mask"]
    
    # Find the position of answer separator
    sep_pos = None
    for i in range(len(input_ids) - len(answer_sep_ids) + 1):
        if input_ids[i:i+len(answer_sep_ids)] == answer_sep_ids:
            sep_pos = i + len(answer_sep_ids)  # position after the separator
            break
    
    # Create labels: -100 for tokens before answer, copy answer tokens
    labels = [-100] * len(input_ids)
    if sep_pos is not None:
        # For answer tokens (from sep_pos to end), keep them as labels
        for i in range(sep_pos, len(input_ids)):
            labels[i] = input_ids[i]
    else:
        # If separator not found, mask everything (should not happen)
        print("Warning: separator not found in:", example["text"][:100])
    
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

# Apply to dataset
tokenized_dataset = dataset.map(tokenize_and_mask, batched=False, remove_columns=["text"])
print("Sample label distribution (non -100):", sum(1 for x in tokenized_dataset[0]["labels"] if x != -100))

# Verify that labels are not all -100
sample_labels = tokenized_dataset[0]["labels"]
if all(l == -100 for l in sample_labels):
    raise RuntimeError("All labels are -100! Check answer separator matching.")
else:
    print("✅ Labels are correct. Answer tokens present.")

In [ ]:
# Split into test (100 samples) and train (remaining)
test_dataset = tokenized_dataset.select(range(100))
train_dataset = tokenized_dataset.select(range(100, len(tokenized_dataset)))  # all remaining

print(f"Test samples: {len(test_dataset)}, Training samples: {len(train_dataset)}")

# Collator for padding (same as before)
def collator(batch):
    input_ids = [torch.tensor(item["input_ids"], dtype=torch.long) for item in batch]
    attention_mask = [torch.tensor(item["attention_mask"], dtype=torch.long) for item in batch]
    labels = [torch.tensor(item["labels"], dtype=torch.long) for item in batch]
    
    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = torch.nn.utils.rnn.pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)
    
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

test_loader = DataLoader(test_dataset, batch_size=2, shuffle=True, collate_fn=collator)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collator)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.train()

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.01)
total_steps = 20
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=5, num_training_steps=total_steps)

print("🚀 Running test training (20 steps)...")
step = 0
progress = tqdm(total=total_steps)
losses = []

while step < total_steps:
    for batch in test_loader:
        if step >= total_steps:
            break
        batch = {k: v.to(device) for k, v in batch.items()}
        
        outputs = model(**batch)
        loss = outputs.loss
        
        # Check loss is tensor, not int
        if isinstance(loss, int):
            raise RuntimeError(f"Loss is int ({loss}) - all labels likely -100")
        
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        
        losses.append(loss.item())
        if step % 5 == 0:
            print(f"Step {step}, loss = {loss.item():.4f}")
        
        step += 1
        progress.update(1)
        
        torch.cuda.empty_cache()
        gc.collect()

progress.close()
print("✅ Test training completed!")
print(f"Final loss: {losses[-1]:.4f}")

### Full Training Run
Only run this once the test run succeeds. Takes ~2-3 hours on T4.

In [ ]:
# Full training (3 epochs) using train_loader
optimizer_full = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.01)
num_epochs = 3
total_steps_full = len(train_loader) * num_epochs
scheduler_full = get_linear_schedule_with_warmup(optimizer_full, num_warmup_steps=50, num_training_steps=total_steps_full)

model.train()
print(f"🚀 Starting full training ({num_epochs} epochs)...")
global_step = 0
for epoch in range(num_epochs):
    epoch_loss = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for batch in progress_bar:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer_full.step()
        scheduler_full.step()
        optimizer_full.zero_grad()
        
        epoch_loss += loss.item()
        global_step += 1
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})
        
        if global_step % 50 == 0:
            torch.cuda.empty_cache()
            gc.collect()
    
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1} completed. Average loss: {avg_loss:.4f}")

print("✅ Full training completed!")

In [ ]:
# ----- Save and zip adapter -----
model.save_pretrained("telerag_lora_model")
tokenizer.save_pretrained("telerag_lora_model")
!zip -r telerag_lora_model.zip telerag_lora_model/
print("📦 Adapter saved as `telerag_lora_model.zip` - download from Kaggle output.")

# (Optional) Push to Hugging Face
model.push_to_hub("chaitanya-k/TeleRAG-LoRA", token=hf_token)
tokenizer.push_to_hub("chaitanya-k/TeleRAG-LoRA", token=hf_token)